In [1]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 23.8 MB/s eta 0:00:00


In [15]:
from pypdf import PdfReader

pdf_path = "/content/ML DATA PDF (1).pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print(text[:2000])

1.  What  is  Machine  Learning?   
Machine  Learning  (ML)  is  a  branch  of  Artificial  Intelligence  that  enables  computers  to  learn  patterns  
from
 
data
 
and
 
make
 
predictions
 
or
 
decisions
 
without
 
being
 
explicitly
 
programmed
 
for
 
every
 
task.
 
 
2.  What  are  the  main  types  of  Machine  Learning?   The  three  main  types  are  Supervised  Learning,  Unsupervised  Learning,  and  Reinforcement  Learning.  
Supervised
 
learning
 
uses
 
labeled
 
data,
 
unsupervised
 
learning
 
finds
 
patterns
 
in
 
unlabeled
 
data,
 
and
 
reinforcement
 
learning
 
learns
 
through
 
rewards
 
and
 
penalties.
 
 
3.  What  is  Supervised  Learning?   Supervised  Learning  is  a  type  of  ML  in  which  a  model  learns  from  labeled  training  data.  The  input  data  
and
 
the
 
correct
 
output
 
are
 
provided
 
during
 
training.
 
Examples
 
include
 
house-price
 
prediction
 
and
 
spam
 
email
 
classification.
 
 
4.  What  is  Unsupervised  Lea

In [17]:
chunk_size = 300
chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])

print("Number of chunks:", len(chunks))
print(chunks[0])

Number of chunks: 16
1.  What  is  Machine  Learning?   
Machine  Learning  (ML)  is  a  branch  of  Artificial  Intelligence  that  enables  computers  to  learn  patterns  
from
 
data
 
and
 
make
 
predictions
 
or
 
decisions
 
without
 
being
 
explicitly
 
programmed
 
for
 
every
 
task.
 
 
2.  What  are  the  


In [18]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(16, 384)


In [19]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Vectors stored:", index.ntotal)

Vectors stored: 16


In [22]:
def ask_rag(question):

    # Convert question into embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # Retrieve relevant chunks
    k = 2
    distances, indices = index.search(
        question_embedding,
        k
    )

    retrieved_chunks = [
        chunks[i] for i in indices[0]
    ]

    # Create context
    context = "\n\n".join(retrieved_chunks)

    # Prompt
    prompt = f"""Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:"""

    # Generate answer
    result = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False,
        return_full_text=False
    )

    answer = result[0]["generated_text"]

    return answer.strip()

In [24]:
ask_rag("What is  linear regression?")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Linear regression is a type of supervised learning technique used to predict continuous numerical values. It is particularly useful for predicting house prices, temperature, sales, or student grades based on various features like age, income, and location. The model learns patterns from the data and makes predictions or decisions without being explicitly programmed for every task. This method helps in understanding relationships between variables and making accurate predictions about future outcomes.'

In [31]:
%%writefile app.py

import streamlit as st
import faiss
import re

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline



st.set_page_config(
    page_title="PDF Question Answering",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 AI Research Assistant")
st.write("Upload a PDF and ask questions from its content.")




@st.cache_resource
def load_embedding_model():

    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    return model



@st.cache_resource
def load_llm():

    model = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-0.5B-Instruct"
    )

    return model


embedding_model = load_embedding_model()
generator = load_llm()



def process_pdf(uploaded_file):

    reader = PdfReader(uploaded_file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"


    text = text.replace("\n", " ")
    text = " ".join(text.split())


    chunks = re.split(
        r'(?=\d+\.\s)',
        text
    )


    chunks = [
        chunk.strip()
        for chunk in chunks
        if chunk.strip()
    ]

    return chunks


def create_index(chunks):

    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True
    ).astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    return index




def ask_rag(question, chunks, index):


    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")


    k = min(2, len(chunks))


    distances, indices = index.search(
        question_embedding,
        k
    )


    retrieved_chunks = [
        chunks[i]
        for i in indices[0]
    ]


    context = "\n\n".join(
        retrieved_chunks
    )


    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""


    result = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False,
        return_full_text=False
    )

    answer = result[0]["generated_text"]

    return answer.strip(), retrieved_chunks



uploaded_file = st.file_uploader(
    "📄 Upload your PDF",
    type=["pdf"]
)



if uploaded_file:

    # Process only once
    if (
        "chunks" not in st.session_state
        or st.session_state.get("file_name") != uploaded_file.name
    ):

        with st.spinner("Processing PDF..."):

            chunks = process_pdf(
                uploaded_file
            )

            index = create_index(
                chunks
            )

            st.session_state.chunks = chunks
            st.session_state.index = index
            st.session_state.file_name = uploaded_file.name

    st.success(
        f"PDF processed successfully! "
        f"{len(st.session_state.chunks)} chunks created."
    )

    question = st.text_input(
        "💬 Ask a question about your PDF:"
    )


    if st.button("Ask"):

        if question:

            with st.spinner(
                "Searching and generating answer..."
            ):

                answer, sources = ask_rag(
                    question,
                    st.session_state.chunks,
                    st.session_state.index
                )


            st.subheader("Answer")

            st.write(answer)


            # Retrieved context
            st.subheader("📚 Retrieved Context")

            for i, source in enumerate(sources):

                with st.expander(
                    f"Source {i + 1}"
                ):

                    st.write(source)

        else:

            st.warning(
                "Please enter a question."
            )

Overwriting app.py


In [32]:
%%writefile requirements.txt

streamlit
transformers
sentence-transformers
faiss-cpu
pypdf
torch
accelerate

Overwriting requirements.txt


In [34]:
%%writefile README.md

# 🤖 AI Research Assistant

A simple RAG-based application that lets users upload PDF documents and ask questions about their content.

## Features

- PDF upload and text extraction
- Text chunking
- Semantic embeddings
- FAISS similarity search
- Qwen LLM-based answers
- Streamlit interface

## RAG Pipeline

PDF → Text Extraction → Chunking → Embeddings → FAISS → Context Retrieval → Qwen LLM → Answer

## Technologies

- Python
- Streamlit
- Hugging Face Transformers
- Sentence Transformers
- FAISS
- PyPDF
- PyTorch

## Models

**Embedding:** `sentence-transformers/all-MiniLM-L6-v2`

**LLM:** `Qwen/Qwen2.5-0.5B-Instruct`

## How It Works

1. Upload a PDF.
2. Extract and split the text.
3. Generate embeddings and store them in FAISS.
4. Ask a question.
5. Retrieve relevant content.
6. Generate an answer using Qwen.

## Purpose

A beginner friendly AI Research Assistant demonstrating a complete RAG pipeline for document based question answering.

Overwriting README.md


In [33]:
%%writefile .gitignore

__pycache__/
*.pyc
.venv/
venv/
env/
.ipynb_checkpoints/
.cache/

Overwriting .gitignore
